# Project 4: Donut Fine-Tuning for Delivery Note Information Extraction
### Halden Industrie GmbH - Document Intelligence Engine (P4 M2)

This self-contained notebook fine-tunes **`naver-clova-ix/donut-base`** (Document Understanding Transformer) on bilingual (DE/EN) synthetic delivery notes to extract structured key-value pairs without an OCR pre-pass.

- **Hardware target**: Kaggle Free GPU (Tesla T4 or P100, 16GB VRAM)
- **Architecture**: Swin-B vision encoder + mBART decoder (End-to-End OCR-free)
- **Resolution**: $1280 \times 960$ input page resolution
- **Precision**: FP16 mixed precision with gradient accumulation
- **Evaluation**: Field-level Macro/Micro F1 & Tree Edit Distance Accuracy

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers datasets evaluate sentencepiece accelerate albumentations zstandard
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Active device: {device} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device name: {torch.cuda.get_device_name(0)}')
    print(f'Memory allocated: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB')

### Step 2: Load Dataset & Target XML Schema
Each document image is paired with a ground-truth JSON containing `gt_parse` fields:
- `supplier_name`, `supplier_address`, `delivery_note_no`, `po_number`, `delivery_date`
- `line_items`: `[{part_no, description, qty, unit}]`
- `total_qty`

Templates 1-4 are used for training/validation, while templates 5-6 are held out for unseen evaluation.

In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel

MODEL_ID = 'naver-clova-ix/donut-base'
processor = DonutProcessor.from_pretrained(MODEL_ID)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID)

# Configure input and output sequence constraints
IMAGE_SIZE = [1280, 960]
processor.image_processor.size = {'height': IMAGE_SIZE[0], 'width': IMAGE_SIZE[1]}
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id

# Register task and semantic XML tokens
SPECIAL_TOKENS = [
    '<s_delivery_note>', '</s_delivery_note>',
    '<s_supplier_name>', '</s_supplier_name>',
    '<s_supplier_address>', '</s_supplier_address>',
    '<s_delivery_note_no>', '</s_delivery_note_no>',
    '<s_po_number>', '</s_po_number>',
    '<s_delivery_date>', '</s_delivery_date>',
    '<s_line_items>', '</s_line_items>',
    '<s_item>', '</s_item>',
    '<s_part_no>', '</s_part_no>',
    '<s_description>', '</s_description>',
    '<s_qty>', '</s_qty>',
    '<s_unit>', '</s_unit>',
    '<s_total_qty>', '</s_total_qty>',
]
processor.tokenizer.add_special_tokens({'additional_special_tokens': SPECIAL_TOKENS})
model.decoder.resize_token_embeddings(len(processor.tokenizer))
print(f'Vocabulary size after adding special tokens: {len(processor.tokenizer)}')

### Step 3: Dataset Definition & Tokenization Pipeline

In [ ]:
class DonutDeliveryDataset(torch.utils.data.Dataset):
    def __init__(self, data_list, processor, max_length=512):
        self.data_list = data_list
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        img = item['image']
        if isinstance(img, str):
            img = Image.open(img).convert('RGB')

        pixel_values = self.processor(img, return_tensors='pt').pixel_values.squeeze(0)
        target_seq = item['target_sequence']
        labels = self.processor.tokenizer(
            target_seq,
            add_special_tokens=False,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        ).input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels}

print('Dataset class ready.')

### Step 4: Training Loop with Evaluation on Held-Out Templates

In [ ]:
from transformers import Seq2SeqTrainingArguments

OUTPUT_DIR = '/kaggle/working/donut_delivery_note'

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=8,
    learning_rate=3e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=1,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_total_limit=2,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    predict_with_generate=True,
    generation_max_length=512,
    metric_for_best_model='f1',
    load_best_model_at_end=True,
    dataloader_num_workers=2,
)

print('Training arguments configured.')

### Step 5: Inference Benchmark (CPU vs CUDA)
Evaluates latency and accuracy on seen templates vs completely unseen templates (Templates 5 & 6).

In [ ]:
def benchmark_inference(model, processor, sample_images, device='cpu'):
    model.to(device)
    model.eval()
    latencies = []
    for img in sample_images:
        t0 = time.perf_counter()
        pixel_values = processor(img, return_tensors='pt').pixel_values.to(device)
        task_prompt = '<s_delivery_note>'
        decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors='pt').input_ids.to(device)
        with torch.no_grad():
            _ = model.generate(
                pixel_values,
                decoder_input_ids=decoder_input_ids,
                max_length=512,
                num_beams=1,
                return_dict_in_generate=True,
            )
        latencies.append(time.perf_counter() - t0)
    return np.mean(latencies), np.median(latencies), np.percentile(latencies, 95)

print('Benchmark harness defined.')

### Step 6: Export Turnkey Artifacts
Saves the fine-tuned model and processor ready for local air-gapped CPU deployment.

In [ ]:
final_save_path = Path(OUTPUT_DIR) / 'final_best'
final_save_path.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(final_save_path))
processor.save_pretrained(str(final_save_path))
print(f'Model and processor successfully saved to {final_save_path}')